# 01b · Download validation data for the season dates

Independent dates to check the Sentinel-2 season dates of `03` against:

1. **MeteoSwiss phenology network** (open data, no account): European larch *needle colouring 50 %* and *needle drop 50 %*, observed by eye at stations near the AOI. Colouring 50 % corresponds to the yellowing peak / midpoint, needle drop 50 % to the end of the season. The stations lie lower than most of the AOI, so `03` compares year-to-year anomalies, not absolute dates.
2. **Copernicus HR-VPP** (Sentinel-2, 10 m, 2017–): start and end of season per pixel, from a TIMESAT double-logistic fit to the Plant Phenology Index; end of season = 15 % of the seasonal amplitude (HR-VPP ATBD, Tian et al. 2021). Downloading needs a free WEkEO account and the `hda` package (see section 2).

- **Settings:** `config.py` (`VALIDATION_RADIUS_KM`, `PHENO_PARAMS`, `HRVPP_PRODUCTS`, years)
- **Reads:** `cache/<AOI>_zones/zones.tif` (from `01_download_sentinel2`)
- **Writes** to `cache/validation/<AOI>/`:
  - `meteoswiss_larch.csv`: one row per station, year and phase
  - `meteoswiss_stations.csv`: the stations used
  - `hrvpp_by_zone.csv`: HR-VPP dates per zone and year (median and quartiles of the zone's pixels)
- **Next:** `03_phenology_metrics` (section 7 compares all methods with these data)

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from pyproj import Transformer

from config import *
from phenology_utils import collection_asset, download, load_zone_table, stac_items

PHENO_COLLECTION = "ch.meteoschweiz.ogd-phenology"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

zones_tif = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
x0, y0, x1, y1 = zones_tif.rio.bounds()
to_wgs84 = Transformer.from_crs(zones_tif.rio.crs, "EPSG:4326", always_xy=True)
AOI_LON, AOI_LAT = to_wgs84.transform((x0 + x1) / 2, (y0 + y1) / 2)
AOI_BBOX = [*to_wgs84.transform(x0, y0), *to_wgs84.transform(x1, y1)]   # lon/lat: west, south, east, north
print(f"AOI: {AOI_NAME} | centre {AOI_LAT:.3f} N, {AOI_LON:.3f} E | writes to {VALIDATION_DIR}")

## 1. MeteoSwiss phenology observations (European larch)

The Swiss Phenology Network (about 160 stations) records the date when 50 % of the larch needles at a site are coloured (`mlard94d`) or have fallen (`mlard95d`). Data: MeteoSwiss open data, collection `ch.meteoschweiz.ogd-phenology` (one CSV per station, one column per phase, dates as YYYYMMDD).

Stations within `VALIDATION_RADIUS_KM` of the AOI centre that observe one of `PHENO_PARAMS` are kept.

In [ ]:
def read_meta(name):
    path = download(collection_asset(PHENO_COLLECTION, name), VALIDATION_DIR / "meteoswiss" / name)
    return pd.read_csv(path, sep=";", encoding="cp1252")


stations = read_meta("ogd-phenology_meta_stations.csv").set_index("station_abbr")
inventory = read_meta("ogd-phenology_meta_datainventory.csv")
observing = inventory[inventory.parameter_shortname.isin(PHENO_PARAMS)].station_abbr.unique()
lat, lon = stations.station_coordinates_wgs84_lat, stations.station_coordinates_wgs84_lon
stations["distance_km"] = 6371 * np.hypot(np.radians(lat - AOI_LAT), np.radians(lon - AOI_LON) * np.cos(np.radians(AOI_LAT)))
near = stations.loc[observing].query("distance_km <= @VALIDATION_RADIUS_KM").sort_values("distance_km")
near = near[["station_name", "station_height_masl", "distance_km", "station_coordinates_wgs84_lat",
             "station_coordinates_wgs84_lon"]].rename(columns=lambda c: c.replace("station_", "").replace("coordinates_wgs84_", ""))
print(f"{len(near)} stations within {VALIDATION_RADIUS_KM} km observe larch needle colouring or drop")

# One CSV per station (STAC item id = lower-case abbreviation)
assets = {item["id"].upper(): next(iter(item["assets"].values()))["href"] for item in stac_items(PHENO_COLLECTION)}
rows = []
for abbr in near.index:
    path = download(assets[abbr], VALIDATION_DIR / "meteoswiss" / f"{abbr.lower()}_y.csv")
    data = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    year = pd.to_datetime(data.reference_timestamp, format="%d.%m.%Y %H:%M").dt.year
    for param, phase in PHENO_PARAMS.items():
        if param not in data:
            continue
        when = pd.to_datetime(data[param], format="%Y%m%d", errors="coerce")
        ok = when.notna()
        rows.append(pd.DataFrame({"station": abbr, "year": year[ok], "param": param, "phase": phase,
                                  "date": when[ok].dt.date, "doy": when[ok].dt.dayofyear}))
larch_obs = pd.concat(rows, ignore_index=True).join(near[["name", "height_masl", "distance_km"]], on="station")
larch_obs = larch_obs[larch_obs.year.between(FIRST_YEAR - 10, LAST_YEAR)]   # some years before for the station means
near = near.join(larch_obs[larch_obs.year >= FIRST_YEAR].pivot_table(index="station", columns="param", values="year",
                                                                    aggfunc="nunique").add_prefix("years_"))
near = near.dropna(how="all", subset=[c for c in near if c.startswith("years_")])   # no observations since FIRST_YEAR
display(near.round(1))

larch_obs.to_csv(VALIDATION_DIR / "meteoswiss_larch.csv", index=False)
near.to_csv(VALIDATION_DIR / "meteoswiss_stations.csv")

fig, axes = plt.subplots(1, len(PHENO_PARAMS), figsize=(12, 4), sharey=True)
for ax, (param, phase) in zip(axes, PHENO_PARAMS.items()):
    for abbr, g in larch_obs[(larch_obs.param == param) & (larch_obs.year >= FIRST_YEAR)].groupby("station"):
        ax.plot(g.year, g.doy, "o-", ms=4, label=f"{abbr} ({near.height_masl[abbr]:.0f} m)")
    ax.set_title(f"Larch {phase}")
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("day of year")
axes[-1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 2. Copernicus HR-VPP start and end of season

HR-VPP (High Resolution Vegetation Phenology and Productivity, Copernicus Land Monitoring Service) gives, per 10 m pixel and year, the start (`SOSD`) and end (`EOSD`) of up to two seasons; season 1 is used. It is distributed through WEkEO's Harmonized Data Access (HDA) API:

1. Create a free account at https://www.wekeo.eu
2. Install the client in the project environment: `pip install hda`
3. Store the login in `~/.hdarc` (two lines: `user:<name>` and `password:<password>`)

Without a login this section is skipped and `03` compares with MeteoSwiss only. The yearly tiles are large (Sentinel-2 tiles, 110 km), so each file is clipped to the zone grid right after the download and then deleted (`KEEP_RAW = False`).

Dates in the rasters are coded as YYDOY (e.g. 21290 = day 290 of 2021); they are converted to the day of the product year (> 365 = next year). The reader also accepts plain DOY values and checks the decoded year.

In [ ]:
HRVPP_DATASET = "EO:EEA:DAT:CLMS_HRVPP_VPP"
RAW_DIR = VALIDATION_DIR / "hrvpp_raw"
KEEP_RAW = False


def hda_client():
    try:
        from hda import Client
        return Client()   # reads ~/.hdarc
    except Exception as e:   # package or login missing
        print(f"HR-VPP skipped ({e!r}). See the steps above.")
        return None


def decode_dates(values, year):
    # YYDOY (or plain DOY) -> day of `year`; nodata (<= 0 or >= 65535) -> NaN
    v = np.where((values > 0) & (values < 65535), values, np.nan)
    if np.nanmax(v, initial=0) < 1000:
        return v
    yy, doy = 2000 + v // 1000, v % 1000
    years = set(np.unique(yy[np.isfinite(yy)]).astype(int))
    if not years <= {year - 1, year, year + 1}:
        raise ValueError(f"Unexpected years {sorted(years)} in the {year} product: check the date coding")
    return doy + 365 * (yy - year)


def zone_stats(doy_grid, year, product):
    # Median and quartiles of the decoded dates of every zone ("all" = every AOI pixel with a zone band)
    names = load_zone_table().index
    zone_id, in_aoi = zones_tif.sel(band=4).values, np.isfinite(zones_tif.sel(band=2).values)
    rows = []
    for zone in names:
        mask = in_aoi if zone == "all" else zone_id == list(names).index(zone)
        d = doy_grid[mask]
        ok = np.isfinite(d)
        q = np.percentile(d[ok], [25, 50, 75]) if ok.any() else [np.nan] * 3
        rows.append({"zone": zone, "year": year, "product": product, "doy_p25": q[0], "doy_median": q[1],
                     "doy_p75": q[2], "valid_share": ok.mean(), "n_pixels": int(mask.sum())})
    return rows


def hrvpp_year(client, year, product):
    query = {"dataset_id": HRVPP_DATASET, "productType": product, "productGroupId": "s1", "bbox": AOI_BBOX,
             "startdate": f"{year}-01-01T00:00:00.000Z", "enddate": f"{year}-12-31T23:59:59.999Z",
             "itemsPerPage": 50, "startIndex": 0}
    matches = client.search(query)
    folder = RAW_DIR / f"{product}_{year}"
    matches.download(download_dir=str(folder))
    files = [f for f in folder.rglob("*.tif") if re.search(rf"{year}.*{product}|{product}.*{year}", f.name)]
    if not files:
        print(f"  {year} {product}: no files")
        return []
    # Tiles overlap at their edges with identical values: the first valid value per pixel is kept
    grid = None
    for f in files:
        # Masked float, so the file's nodata becomes NaN and cannot clash with the fill value of the reprojection
        tile = rioxarray.open_rasterio(f, masked=True).squeeze("band", drop=True).astype("float32")
        on_grid = tile.rio.reproject_match(zones_tif.sel(band=1), resampling=0, nodata=np.nan).values.astype(float)
        dates = decode_dates(on_grid, year)
        grid = dates if grid is None else np.where(np.isfinite(grid), grid, dates)
    if not KEEP_RAW:
        for f in folder.rglob("*"):
            if f.is_file():
                f.unlink()
    return zone_stats(grid, year, product)


client = hda_client()
if client is not None:
    out_file = VALIDATION_DIR / "hrvpp_by_zone.csv"
    done = pd.read_csv(out_file) if out_file.exists() else pd.DataFrame(columns=["year", "product"])
    rows = done.to_dict("records")
    for year in [y for y in YEARS if y < pd.Timestamp.today().year]:   # products appear in the next spring
        for product in HRVPP_PRODUCTS:
            if ((done.year == year) & (done["product"] == product)).any():
                continue
            print(f"{year} {product}")
            try:
                rows += hrvpp_year(client, year, product)
            except Exception as e:   # e.g. the year is not published yet
                print(f"  failed: {e!r}")
    hrvpp = pd.DataFrame(rows)
    hrvpp.to_csv(out_file, index=False)
    if not hrvpp.empty:
        display(hrvpp[hrvpp.zone == "all"].pivot_table(index="year", columns="product", values="doy_median"))